(produce factually incorrect or nonsensical information) or provide outdated information due to their static training data. RAG addresses these limitations by allowing LLMs to access and incorporate up-to-date, external knowledge during the generation process.

**How RAG Works:**

A RAG system typically operates in two main stages:

1.  **Retrieval**: Given a user query, the system first retrieves relevant documents or passages from a vast external knowledge base (e.g., a collection of articles, books, or databases).
2.  **Generation**: The retrieved information is then fed as context to an LLM, which uses this context to generate a more accurate, informed, and attributable response.

This process ensures that the LLM's output is grounded in factual evidence, reducing hallucinations and improving the overall quality and trustworthiness of the generated text.

---

## 2. Environment Setup and Dependencies

First, we need to set up our Google Colab environment by installing the necessary libraries. We will use `pip` to install `transformers`, `torch`, `langchain`, `faiss-cpu`, `sentence-transformers`, and `ragas`.

In [1]:
!pip install -q transformers torch langchain faiss-cpu sentence-transformers ragas accelerate matplotlib pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

After installation, it's good practice to restart the runtime to ensure all packages are loaded correctly.

In [4]:
import os
import torch
from transformers import AutoTokenizer, AutoModel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevance, context_recall, context_precision

print("All necessary libraries imported successfully!")

/tmp/ipykernel_974/2694510013.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


ModuleNotFoundError: No module named 'langchain.vectorstores'

---

## 3. Core Components of a RAG System

Let's break down the essential components that make up a RAG system.

### 3.1. Document Loading

The first step is to load our raw data. This could be text files, PDFs, web pages, etc. For this tutorial, we'll use a simple text file.

In [ ]:
# Create a dummy document for demonstration
dummy_text = """
Retrieval-Augmented Generation (RAG) is an AI framework that enhances the output of large language models (LLMs) by retrieving facts from external knowledge bases to ground the models on the most accurate and up-to-date information. This helps to reduce hallucinations and improve the relevancy of the generated text. RAG systems typically involve two main phases: retrieval and generation. In the retrieval phase, relevant documents or passages are fetched from a knowledge base based on a user's query. In the generation phase, an LLM uses this retrieved information as context to formulate a response. LangChain is a popular framework for building applications with LLMs, including RAG systems. FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It is commonly used as a vector store in RAG architectures.
"""

with open("rag_educational_repo/data/sample_document.txt", "w") as f:
    f.write(dummy_text)

from langchain.document_loaders import TextLoader

loader = TextLoader("rag_educational_repo/data/sample_document.txt")
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")
print(documents[0].page_content[:200] + "...")

### 3.2. Text Splitting

Large documents need to be broken down into smaller, manageable chunks. This is crucial for efficient retrieval, as embedding large texts can be computationally expensive and may exceed the context window of LLMs. `RecursiveCharacterTextSplitter` is a good choice for this.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Split document into {len(chunks)} chunks.")
print(chunks[0].page_content)

### 3.3. Embedding Models

Embedding models convert text chunks into numerical vector representations (embeddings). These embeddings capture the semantic meaning of the text, allowing for semantic search. We'll use a pre-trained model from Hugging Face.

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

# Example of creating an embedding
# text_embedding = embeddings.embed_query("What is RAG?")
# print(f"Embedding dimension: {len(text_embedding)}")

### 3.4. Vector Stores

A vector store (or vector database) is where we store our text embeddings and efficiently search for similar vectors. FAISS is a popular choice for its speed and efficiency.

In [ ]:
vector_store = FAISS.from_documents(chunks, embeddings)

print("Vector store created successfully!")

### 3.5. Retrievers

The retriever is responsible for fetching the most relevant document chunks from the vector store based on a given query. It uses the embeddings to find semantically similar chunks.

In [ ]:
retriever = vector_store.as_retriever()

# Example retrieval
query = "What is Retrieval-Augmented Generation?"
retrieved_docs = retriever.get_relevant_documents(query)

print(f"Retrieved {len(retrieved_docs)} documents for the query: '{query}'")
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)

### 3.6. Generative Models (LLMs)

Finally, we need a Large Language Model to generate the answer using the retrieved context. For this Colab, we'll use a smaller, open-source LLM from Hugging Face that can run on a free GPU.

In [ ]:
# Load a smaller LLM for demonstration in Colab
# You might need to adjust the model based on available GPU memory
model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

from transformers import pipeline

pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    torch_dtype=torch.float16,
    device_map="auto",
)

llm = HuggingFacePipeline(pipeline=pipeline)

print(f"LLM '{model_id}' loaded successfully!")

---

## 4. Building a Basic RAG Pipeline

Now, let's combine all these components into a simple RAG pipeline using LangChain's `RetrievalQA` chain.

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # "stuff" all retrieved documents into the prompt
    retriever=retriever,
    return_source_documents=True
)

# Test the RAG pipeline
query = "What is the purpose of RAG systems?"
result = qa_chain({"query": query})

print(f"Query: {query}")
print(f"\nAnswer: {result['result']}")
print(f"\nSource Documents:")
for i, doc in enumerate(result['source_documents']):
    print(f"--- Document {i+1} ---")
    print(doc.page_content)

---

## 5. Advanced RAG Techniques

To further improve the performance and accuracy of our RAG system, we can employ advanced techniques.

### 5.1. Query Expansion

Query expansion involves generating multiple reformulations or related queries from the original user query. This can help retrieve a broader and potentially more relevant set of documents.

In [ ]:
# This is a conceptual example. In a real scenario, you'd use an LLM
# to generate expanded queries or a more sophisticated technique.

def expand_query(original_query):
    expanded_queries = [
        original_query,
        f"Tell me more about {original_query.lower()}",
        f"Key concepts related to {original_query.lower()}"
    ]
    return expanded_queries

original_query = "What is FAISS?"
expanded_queries = expand_query(original_query)
print(f"Original Query: {original_query}")
print(f"Expanded Queries: {expanded_queries}")

# You would then retrieve documents for each expanded query and combine them.

### 5.2. Re-ranking

After initial retrieval, some documents might be less relevant than others. Re-ranking uses a more sophisticated model (often a cross-encoder) to score the relevance of each retrieved document to the query, and then reorders them. This ensures that the most pertinent information is presented to the LLM.

In [ ]:
# This is a conceptual example. Re-ranking typically involves a separate model.

def rerank_documents(query, documents):
    # In a real implementation, a cross-encoder model would score each doc-query pair.
    # For demonstration, we'll just simulate a re-ordering.
    print("Simulating re-ranking...")
    # Example: simple heuristic (e.g., prioritize docs containing specific keywords)
    reranked_docs = sorted(documents, key=lambda x: query.lower() in x.page_content.lower(), reverse=True)
    return reranked_docs

# retrieved_docs from previous step
reranked_docs = rerank_documents(query, retrieved_docs)

print("\nRe-ranked Documents:")
for i, doc in enumerate(reranked_docs):
    print(f"--- Document {i+1} ---")
    print(doc.page_content)

---

## 6. Evaluation of RAG Systems with RAGAS

Evaluating RAG systems is crucial to ensure they are performing as expected. RAGAS is a framework that helps evaluate various aspects of a RAG pipeline without requiring human annotations for every data point. It focuses on metrics like faithfulness, answer relevance, context recall, and context precision.

In [ ]:
# RAGAS requires a dataset of questions, ground truths, contexts, and answers.
# For a full evaluation, you would prepare a more extensive dataset.

# Dummy data for RAGAS evaluation demonstration
from datasets import Dataset

data = {
    "question": ["What is RAG?", "What is FAISS used for?"],
    "answer": ["RAG enhances LLMs by retrieving external facts to reduce hallucinations and improve relevancy.", "FAISS is a library for efficient similarity search and clustering of dense vectors, often used as a vector store in RAG."],
    "contexts": [
        ["Retrieval-Augmented Generation (RAG) is an AI framework that enhances the output of large language models (LLMs) by retrieving facts from external knowledge bases to ground the models on the most accurate and up-to-date information. This helps to reduce hallucinations and improve the relevancy of the generated text."],
        ["FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It is commonly used as a vector store in RAG architectures."]
    ],
    "ground_truth": [
        "RAG improves LLM outputs by using external knowledge to provide accurate and relevant responses, reducing factual errors.",
        "FAISS is a library for fast similarity search on dense vectors, frequently employed as a vector database in RAG systems."
    ]
}

dataset = Dataset.from_dict(data)

# Generate answers and contexts using our RAG pipeline for the evaluation dataset
# In a real scenario, you would run your qa_chain on a test set to get these.
# For this demo, we'll use the dummy data directly.

# You would typically run something like this:
# from tqdm import tqdm
# answers = []
# contexts = []
# for query in tqdm(dataset["question"]):
#     result = qa_chain({"query": query})
#     answers.append(result["result"])
#     contexts.append([doc.page_content for doc in result["source_documents"]])

# dataset = dataset.add_column("answer", answers)
# dataset = dataset.add_column("contexts", contexts)

# Evaluate with RAGAS
result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevance, context_recall, context_precision]
)

print("\nRAGAS Evaluation Results:")
print(result)

# You can access individual metric scores:
# print(f"Faithfulness: {result['faithfulness']}")
# print(f"Answer Relevance: {result['answer_relevance']}")
# print(f"Context Recall: {result['context_recall']}")
# print(f"Context Precision: {result['context_precision']}")

---

## 7. Conclusion and Further Exploration

This notebook has walked you through the essential steps of building and evaluating a RAG system. You've learned about:

-   The core architecture and components of RAG.
-   How to implement a basic RAG pipeline using LangChain, Hugging Face Transformers, and FAISS.
-   Conceptual understanding of advanced techniques like query expansion and re-ranking.
-   How to evaluate your RAG system using RAGAS.

**Further Steps:**

-   Experiment with different embedding models and LLMs.
-   Explore various text splitting strategies and chunk sizes.
-   Integrate more complex data sources (e.g., PDFs, web scraping).
-   Implement actual query expansion and re-ranking models.
-   Deep dive into RAGAS metrics and interpret the results to improve your RAG pipeline.
-   Consider using other vector stores like Pinecone, Weaviate, or ChromaDB.

Happy Hacking!